# Modélisation : Adult Income (UCI)

Dans cette partie, on compare un large panel de classifieurs via le `ModelRegistry` avec cross val sur le train.
On effectue également les confirmations/infirmations des choix de prétraitement via ablations

On garde également tous les modèles peu importe leurs résultats car on fera le choix final une fois l'hyper-optimisation réalisée (car la performance d'un modèle peut drastiquement évoluer avec l'optimisation de ses hyperparamètres)

Encore une fois, le test set reste de côté tout au long de cette étape, et ne servira que pour l'évaluation finale du modèle final hyper-optimisé (on travaille ici avec des train set et val set)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt

from adult_income import (
    DataLoader,
    ModelEvaluator,
    ModelRegistry,
    PreprocessingPipeline,
)

print("Imports OK")

## 1. Données & preprocessing

In [ ]:
loader = DataLoader()
X_train, y_train = loader.load_train()

pp = PreprocessingPipeline(scaling=True) # pipeline par defaut

# Encodage de la cible (<=50K -> 0, >50K -> 1)
y_train_enc = pp.fit_transform_target(y_train)

print(f"Train : {X_train.shape[0]:,} obs")

## 2. Catalogue de modèles (`ModelRegistry`)

In [ ]:
model_names = ModelRegistry.available()
models = ModelRegistry.get_subset(model_names)
print(f"{len(models)} modèles à comparer :", list(models))

## 3. Validation des choix de prétraitement (ablations)

On vérifie que les propositions de l'AED tiennent face à leurs alternatives,
en CV sur le train (Average Precision)

On utilise deux types de modèles différents, car l'effet d'une transformation dépend du modèle :
- **linéaire** (régression logistique) : sensible aux transfos monotones (`log1p`),
  à la binarisation et à l'encodage 
- **à arbres** (XGBoost) : insensible (voire pénalisée) aux transfos monotones / à l'échelle, et
  capable de trouver elle-même les seuils (zéro/non-zéro)

| Flag | Baseline (AED) | Variante |
|---|---|---|
| `drop_fnlwgt` | `True` (supprimée) | `False` (gardée) |
| `unknown_category` | `True` (`Unknown`) | `False` (mode) |
| `capital_treatment` | `"hurdle"` | `"naive"` |

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import pandas as pd

cv = StratifiedKFold(5, shuffle=True, random_state=42)

probes = {
    "logistique (linéaire)": ModelRegistry.get("logistic_regression"),
    "xgboost (arbres)":      ModelRegistry.get("xgboost"),
}

def ap_par_sonde(build_pp):
    """Average Precision moyenne (CV) de chaque sonde pour une config de prétraitement."""
    return {
        nom: round(cross_val_score(build_pp().build_pipeline(probe), X_train, y_train_enc,
                                   cv=cv, scoring="average_precision", n_jobs=-1).mean(), 4)
        for nom, probe in probes.items()
    }

variants = {
    "baseline (AED)":             lambda: PreprocessingPipeline(), # version pipeline par défaut
    "+ fnlwgt":                   lambda: PreprocessingPipeline(drop_fnlwgt=False),
    "mode au lieu d'Unknown":     lambda: PreprocessingPipeline(unknown_category=False),
    "capital naïf (sans hurdle)": lambda: PreprocessingPipeline(capital_treatment="naive"),
}
ablation = pd.DataFrame({lab: ap_par_sonde(f) for lab, f in variants.items()}).T

ModelEvaluator.plot_grouped_bars(
    ablation, title="Ablations de prétraitement", ylabel="Average Precision (CV)",
)
plt.show()
ablation

## 4. Prétraitement affiné (suite aux ablations)

Les ablations montrent que le hurdle sur `capital-*` dégrade la performance des modèles à
arbres, pour un gain quasi nul sur le linéaire. Le meilleur modèle sera vraisemblablement un arbre car plus puissants en général, donc on
retient le traitement `naive` des `capital-*`. Les autres propositions de l'AED
(drop `fnlwgt`, catégorie `Unknown`, OneHot) sont validées et conservées.

Ce `pp_final` sert pour la comparaison complète et l'hyper-optimisation (notebook séparé).

In [ ]:
pp_final = PreprocessingPipeline(capital_treatment="naive")

pipelines = {name: pp_final.build_pipeline(est) for name, est in models.items()}
print("Prétraitement affiné : capital=naive, occupation=onehot, fnlwgt droppé, catégorie Unknown")
print(f"{len(pipelines)} pipelines prêts pour la comparaison CV")

## 5. Comparaison des modèles (validation croisée)

Tous les modèles sont évalués en CV stratifiée sur le train set (pour garder l'équilibre des classes), avec le
prétraitement affiné `pp_final`. Métrique principale : **Average Precision**
(adaptée au déséquilibre de classes)

In [ ]:
evaluator = ModelEvaluator(cv=5, random_state=42)
evaluator.cross_validate_all(pipelines, X_train, y_train_enc)

evaluator.cv_comparison_table()

In [ ]:
evaluator.plot_cv_comparison(metric="average_precision")
plt.show()

## 5.1 Pondération des classes (vs réglage du seuil)

La cible est déséquilibrée (~24 % de `>50K`). On teste la **pondération des classes** comme
levier, via le paramètre propre à chaque modèle :
- `logistic_regression`, `random_forest` : `class_weight ∈ {None, "balanced"}` 
- `xgboost` : `scale_pos_weight ∈ {1, ratio nég/pos ≈ 3,1}` 
- `gradient_boosting` n'a pas de levier, donc il est écarté (xgboost représente le boosting)

**Méthode :** Probabilités **hors-pli** (CV k-fold train, calcul des performances sur un validation set qui change au fil de la CV). Pour chaque config :
AP/ROC-AUC, puis F1/rappel **au seuil 0,50** et **au seuil optimal de la config** (`seuil*`,
max-F1 **recalculé ici** sur les mêmes probas. Ce seuil est par ailleurs indépendant du
seuil final réglé plus tard dans `hyperopt.ipynb`)

**Conclusion :** L'AP ne change pas beaucoup (en effet, la pondération déplace les probas mais ne réordonne pas), et
une fois chaque config ramenée à son `seuil*`, le `F1@seuil*` est quasi identique avec ou sans
pondération, ce qui fait que c'est un levier **redondant** avec le seuil

On fait donc le choix de ne pas l'activer

In [ ]:
ratio = float((y_train_enc == 0).sum() / (y_train_enc == 1).sum())

weight_probes = [
    {"name": "logistic_regression",
     "pipeline": pp_final.build_pipeline(ModelRegistry.get("logistic_regression")),
     "param": "classifier__class_weight",
     "values": {"aucun": None, "balanced": "balanced"}},
    {"name": "random_forest",
     "pipeline": pp_final.build_pipeline(ModelRegistry.get("random_forest")),
     "param": "classifier__class_weight",
     "values": {"aucun": None, "balanced": "balanced"}},
    {"name": "xgboost",
     "pipeline": pp_final.build_pipeline(ModelRegistry.get("xgboost")),
     "param": "classifier__scale_pos_weight",
     "values": {"aucun": 1.0, f"pondéré (~{ratio:.1f})": ratio}},
]

evaluator.class_weight_comparison(weight_probes, X_train, y_train_enc).round(4)

## 6. Étape suivante : hyper-optimisation

La comparaison ci-dessus classe tous les modèles à hyperparamètres par défaut, mais
aucun n'est écarté ici : présélectionner des modèles finalistes sur les réglages par défaut
serait risqué, car les modèles ne profitent pas tous autant du tuning (le boosting et les
forêts gagnent beaucoup, le linéaire presque rien donc le classement peut donc se réorganiser).

Comme annoncé précédemment, le modèle final sera choisi suite à cette étape d'hyper-optimisation